# Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [2]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
177434,2022-12-29 02:00:00,4,48.47530,35.01600,1.3,-1.2,83.4,0.000,0.0,0.00,...,2,Dnipro,Europe/Kiev,2022-12-29,07:31:42,15:52:18,02:00:00,Дніпропетровська,0,0.0
334936,2023-09-28 13:00:00,19,49.55270,25.58890,17.9,10.2,64.9,0.000,0.0,0.00,...,13,Ternopil,Europe/Kiev,2023-09-28,07:12:16,19:03:46,13:00:00,Тернопільська,0,0.0
489826,2024-06-23 12:00:00,13,49.84440,24.02540,20.0,14.8,74.7,6.000,100.0,4.17,...,12,Lviv,Europe/Kiev,2024-06-23,05:16:06,21:36:24,12:00:00,Львівська,0,0.0
125035,2022-09-29 02:00:00,22,49.41680,26.97430,13.1,9.3,78.6,0.000,0.0,0.00,...,2,Khmelnytskyi,Europe/Kiev,2022-09-29,07:08:31,18:55:35,02:00:00,Хмельницька,0,0.0
615574,2025-01-27 19:00:00,25,51.49370,31.29440,6.0,4.0,86.9,0.000,0.0,0.00,...,19,Chernihiv,Europe/Kiev,2025-01-27,07:40:16,16:35:38,19:00:00,Чернігівська,0,0.0
138305,2022-10-22 03:00:00,20,50.00420,36.23580,4.4,0.3,76.2,0.000,0.0,0.00,...,3,Kharkiv,Europe/Kiev,2022-10-22,07:07:51,17:30:28,03:00:00,Харківська,1,60.0
444333,2024-04-05 12:00:00,24,48.29240,25.93550,11.6,6.0,69.9,0.981,100.0,8.33,...,12,Chernivtsi,Europe/Kiev,2024-04-05,06:45:35,19:53:05,12:00:00,Чернівецька,0,0.0
286690,2023-07-06 19:00:00,13,49.84440,24.02540,21.3,17.4,79.8,2.400,100.0,20.83,...,19,Lviv,Europe/Kiev,2023-07-06,05:23:08,21:33:47,19:00:00,Львівська,0,0.0
19907,2022-03-30 14:00:00,14,46.97336,31.98522,10.0,4.1,70.4,0.000,0.0,0.00,...,14,Mykolaiv,Europe/Kiev,2022-03-30,06:35:37,19:18:23,14:00:00,Миколаївська,0,0.0
195528,2023-01-29 12:00:00,2,49.23360,28.44860,-2.0,-4.7,82.3,0.200,100.0,4.17,...,12,Vinnytsia,Europe/Kiev,2023-01-29,07:42:05,16:57:06,12:00:00,Вінницька,0,0.0


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [6]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[ns]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_s

In [7]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].rolling(window=24, min_periods=1).sum().reset_index(level=0, drop=True)
df_final['alarm_in_last_hour'] = df_final.groupby('region_id')['alarm_active'].shift(1)
df_final['total_active_alarms'] = df_final.groupby('datetime_hour')['alarm_active'].transform('sum')
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night
526921,2024-08-26 22:00:00,3,50.74690,25.32630,23.7,16.4,66.6,0.0,0.0,0.00,...,20:16:06,22:00:00,Волинська,0,0.000000,6.0,0.0,11,0,0
179051,2022-12-31 21:00:00,14,46.97336,31.98522,4.4,2.2,85.9,0.0,0.0,0.00,...,16:12:21,21:00:00,Миколаївська,0,0.000000,5.0,0.0,1,1,0
90334,2022-07-30 20:00:00,25,51.49370,31.29440,20.0,15.4,75.9,1.0,100.0,4.17,...,20:46:19,20:00:00,Чернігівська,0,0.000000,1.0,0.0,1,1,0
113031,2022-09-08 06:00:00,18,50.90800,34.79720,11.1,4.9,67.4,0.0,0.0,0.00,...,19:11:17,06:00:00,Сумська,1,11.933333,5.0,1.0,9,0,1
50584,2022-05-22 20:00:00,19,49.55270,25.58890,12.4,5.5,63.7,0.0,0.0,0.00,...,21:04:28,20:00:00,Тернопільська,0,0.000000,2.0,0.0,0,1,0
120792,2022-09-21 18:00:00,2,49.23360,28.44860,9.9,7.5,86.0,1.4,100.0,29.17,...,19:06:57,18:00:00,Вінницька,0,0.000000,0.0,0.0,0,0,0
516128,2024-08-08 04:00:00,10,50.45060,30.52430,21.3,14.8,68.9,0.0,0.0,0.00,...,20:29:48,04:00:00,Київська,0,0.000000,6.0,0.0,2,0,1
458281,2024-04-29 18:00:00,3,50.74690,25.32630,16.4,5.9,53.6,0.0,0.0,0.00,...,20:37:10,18:00:00,Волинська,0,0.000000,0.0,0.0,9,0,0
482684,2024-06-11 02:00:00,23,49.44070,32.06370,24.2,16.8,66.1,31.5,100.0,45.83,...,20:57:57,02:00:00,Черкаська,0,0.000000,3.0,0.0,8,0,1
232346,2023-04-03 11:00:00,4,48.47530,35.01600,8.1,6.8,91.4,10.0,100.0,8.33,...,19:12:52,11:00:00,Дніпропетровська,0,0.000000,8.0,0.0,3,0,0


In [8]:
"""neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

def get_neighbour_sum_safe(row):
    region_id = row['region_id']
    datetime = row['datetime_hour']
    
    if region_id in neighbouring_regions:
        neighbours = neighbouring_regions[region_id]
        
        if datetime in alarms_matrix.index:
            valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]

            if valid_neighbours:
                return alarms_matrix.loc[datetime, valid_neighbours].sum()
    return 0

df_final['neighbour_alarms'] = df_final.apply(get_neighbour_sum_safe, axis=1)

df_final.sample(15)"""

"neighbouring_regions = {\n    1: [21],\n    2: [6, 10, 11, 15, 22, 23, 24],\n    3: [13, 17],\n    4: [5, 8, 11, 14, 16, 20, 21],\n    5: [4, 8, 12, 20],\n    6: [2, 10, 17, 22],\n    7: [9, 13],\n    8: [4, 5, 21],\n    9: [7, 13, 19, 24],\n    10: [2, 6, 16, 23, 25],\n    11: [2, 4, 14, 15, 16, 23],\n    12: [5, 20],\n    13: [3, 7, 9, 17, 19],\n    14: [4, 11, 15, 21],\n    15: [2, 11, 14],\n    16: [4, 10, 11, 18, 20, 23, 25],\n    17: [3, 6, 13, 19, 22],\n    18: [16, 20, 25],\n    19: [9, 13, 17, 22, 24],\n    20: [4, 5, 12, 16, 18],\n    21: [1, 4, 8, 14],\n    22: [2, 6, 17, 19, 24],\n    23: [2, 10, 11, 16],\n    24: [2, 9, 19, 22],\n    25: [10, 16, 18], \n    26: [10]\n}\n\nalarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)\n\ndef get_neighbour_sum_safe(row):\n    region_id = row['region_id']\n    datetime = row['datetime_hour']\n\n    if region_id in neighbouring_regions:\n        neighbours = neighbouring_

In [9]:
def calculate_hours_since_last(series):
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last)

In [10]:
"""df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)
df_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"""

"df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)\ndf_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)\ndf_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"

In [11]:
df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

In [12]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,hours_since_last_alarm,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12
492505,2024-06-28 04:00:00,3,50.7469,25.3263,25.0,14.5,55.6,0.000,0.0,0.00,...,0.0,0.0,3,0,1,25,0.0,0.0,0.0,0.0
924,2022-02-25 14:00:00,15,46.4725,30.7371,5.3,-0.9,66.0,0.000,0.0,0.00,...,0.0,0.0,2,0,0,38,0.0,0.0,0.0,0.0
279068,2023-06-23 13:00:00,23,49.4407,32.0637,25.7,15.9,57.2,0.000,0.0,0.00,...,3.0,0.0,0,0,0,10,0.0,0.0,0.0,1.0
356426,2023-11-04 21:00:00,4,48.4753,35.0160,14.8,13.3,91.3,0.400,100.0,4.17,...,10.0,0.0,1,1,0,2,0.0,1.0,1.0,0.0
29950,2022-04-17 00:00:00,25,51.4937,31.2944,3.1,0.5,83.6,2.200,100.0,70.83,...,6.0,0.0,8,1,1,2,0.0,1.0,0.0,0.0
242970,2023-04-21 21:00:00,21,46.6401,32.6142,10.7,9.7,93.6,5.000,100.0,8.33,...,3.0,0.0,4,0,0,0,0.0,1.0,0.0,0.0
261439,2023-05-23 23:00:00,9,48.9226,24.7147,16.8,7.1,57.6,0.000,0.0,0.00,...,0.0,0.0,2,0,1,42,0.0,0.0,0.0,0.0
230263,2023-03-30 20:00:00,9,48.9226,24.7147,3.8,-4.9,56.0,0.000,0.0,0.00,...,1.0,0.0,1,0,0,7,0.0,0.0,0.0,0.0
22378,2022-04-03 21:00:00,13,49.8444,24.0254,0.2,-4.0,75.7,3.000,100.0,8.33,...,4.0,1.0,13,1,0,1,1.0,0.0,0.0,0.0
10521,2022-03-14 06:00:00,11,48.5085,32.2656,-0.9,-5.1,74.4,0.000,0.0,0.00,...,11.0,1.0,17,0,1,0,1.0,1.0,1.0,0.0


In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1068,2025-02-06,0.824820,0.149730,-0.019705,0.052894,0.003553,0.010955,-0.020027,0.041350,0.049878,...,0.014954,-0.009188,0.005416,-0.006358,0.033455,0.014359,-0.003580,-0.002307,-0.000500,0.015001
700,2024-01-31,0.752040,-0.026956,-0.223836,0.113146,0.218574,0.174766,-0.117009,-0.120065,0.100284,...,-0.023043,0.013794,0.021722,-0.039436,0.001281,0.026316,0.043503,-0.010345,0.025673,-0.032235
111,2022-06-15,0.664999,-0.226142,0.318017,0.057576,-0.024569,0.070099,0.063617,-0.129224,-0.016716,...,0.009121,0.008315,-0.007130,0.045575,-0.002441,-0.026612,-0.005262,0.004272,-0.015392,-0.035232
175,2022-08-18,0.766331,-0.224936,0.173460,-0.078307,-0.087942,0.098603,0.004250,0.080944,0.174986,...,0.000591,-0.007707,0.007027,-0.016475,0.008277,0.004707,-0.017492,-0.010876,0.004125,-0.002416
434,2023-05-07,0.679060,-0.260342,-0.097518,-0.058915,0.013944,0.068511,-0.034395,0.105898,-0.127467,...,-0.014862,0.018821,-0.018597,0.004451,-0.016457,0.014335,-0.030076,-0.018403,0.010745,-0.024611
931,2024-09-18,0.783229,0.052118,-0.082332,-0.037166,-0.133481,-0.009538,-0.049787,0.009201,0.118328,...,0.018188,0.038479,0.006139,-0.004798,-0.003007,0.016010,0.004578,0.022740,0.000809,-0.008400
1236,2025-07-24,0.737931,0.237151,0.044019,0.033179,-0.157500,0.133262,0.075821,-0.037075,-0.095951,...,0.010427,0.028755,-0.028984,-0.016628,-0.017644,0.036512,-0.012059,0.020592,-0.026710,0.006676
826,2024-06-05,0.805669,-0.046296,-0.136245,0.122048,-0.103417,0.051414,0.074962,-0.095364,-0.041190,...,-0.020440,-0.024189,0.008610,0.008123,-0.023711,-0.001452,0.006708,0.001164,0.027446,0.012852
1289,2025-09-15,0.778117,0.302665,0.063574,-0.112442,-0.042756,0.009964,-0.071932,0.116963,-0.035157,...,0.029624,0.011033,0.031065,0.033263,0.021059,-0.004488,0.011720,-0.024108,0.003302,-0.020609
1254,2025-08-11,0.767085,0.317685,0.040743,-0.014917,-0.116803,0.079460,-0.017638,0.163147,0.072366,...,0.001262,0.008760,0.003224,0.002582,0.014153,0.004517,0.006485,-0.007643,-0.009298,-0.002677


In [14]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [17]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
160815,2022-05-24 10:00:00,8,47.82890,35.16260,16.1,4.8,50.6,0.000,0.0,0.00,...,-0.048013,0.012105,0.024247,-0.017808,0.057977,0.007241,-0.019532,0.000438,-0.017757,0.013846
24125,2024-11-25 08:00:00,2,49.23360,28.44860,-2.3,-4.8,83.6,0.000,0.0,0.00,...,-0.015593,-0.002290,0.005275,0.005251,-0.013160,0.009243,0.025455,0.017300,0.016119,0.000321
131497,2025-01-30 16:00:00,6,50.25360,28.66540,4.9,2.8,86.6,0.000,0.0,0.00,...,0.004606,0.018754,-0.002219,-0.002335,0.030275,0.017571,0.007201,-0.008824,-0.005943,0.005123
375434,2022-09-28 21:00:00,17,50.61927,26.25131,10.4,8.5,88.8,7.053,100.0,4.17,...,0.003845,-0.008455,-0.010394,0.011441,-0.002275,-0.008278,-0.003027,0.029635,-0.017272,0.014798
240626,2022-06-13 06:00:00,11,48.50850,32.26560,24.3,13.0,52.4,0.000,0.0,0.00,...,-0.021881,-0.019308,-0.004401,-0.004038,0.011006,0.000192,-0.028415,0.011660,-0.033678,-0.015724
417205,2024-06-28 13:00:00,18,50.90800,34.79720,21.9,14.3,65.6,0.000,0.0,0.00,...,0.021604,0.020152,-0.006235,0.033250,-0.002626,-0.001964,0.011057,-0.004895,-0.003400,-0.025838
95287,2023-12-20 18:00:00,5,48.00200,37.81450,4.7,4.1,95.8,0.000,0.0,0.00,...,-0.043065,0.016618,-0.005337,0.011406,-0.016974,0.034557,0.018870,-0.001597,-0.025460,-0.012045
78451,2025-01-24 04:00:00,4,48.47530,35.01600,2.4,2.3,99.5,0.700,100.0,4.17,...,0.004342,-0.004341,0.008450,-0.022114,0.013419,-0.003373,-0.015043,-0.009937,-0.008823,0.001754
505650,2022-07-07 04:00:00,22,49.41680,26.97430,17.7,12.0,71.9,1.000,100.0,4.17,...,-0.028819,-0.040007,0.006614,-0.014500,-0.006963,0.018436,-0.023479,-0.016156,-0.026092,0.006419
111902,2022-11-06 03:00:00,6,50.25360,28.66540,7.2,5.4,88.3,2.700,100.0,37.50,...,0.014931,0.020483,-0.013289,0.023865,0.015016,-0.003194,0.002225,0.009971,0.017150,0.040009


In [18]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

In [20]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final["isw_topic_mean"] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = df_final['isw_total_intensity'].diff(24)

df_final["isw_intensity_ema"] = (df_final.groupby("region_id")["isw_total_intensity"].transform(lambda x: x.ewm(span=24).mean()))

df_final["isw_topic_entropy"] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

"""df_final.drop(columns=topic_cols, inplace=True)
df_final = df_final.copy()"""
df_final.fillna(0, inplace=True)

### TELEGRAM MERGE

In [22]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [23]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000991,-0.011395,0.032671,-0.050983,0.061079,...,0.017144,-0.013950,-0.027494,0.034615,-0.032077,-0.051153,-0.004736,-0.004042,-0.052157,-0.026376
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077088,-0.078418,0.091533,...,-0.019095,0.025732,0.000545,0.020312,-0.028054,0.001602,0.006979,-0.064937,0.022653,-0.017796
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053648,-0.076999,0.085929,...,0.025382,-0.003562,-0.043411,-0.007184,-0.009170,0.006113,-0.004557,0.020586,0.010520,0.012833
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059695,...,0.008731,-0.007507,-0.021414,0.028226,-0.025335,-0.045774,-0.001725,0.002281,-0.042259,-0.017379
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089563,-0.088812,0.109835,...,0.017981,0.027449,-0.019751,0.008681,0.023138,-0.009474,-0.030269,0.013652,-0.022735,0.006830


In [24]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

In [25]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]
tg_hourly = (df_tg_matrix.groupby("datetime_hour")[topic_cols].mean().reset_index())

In [26]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000991,-0.011395,0.032671,-0.050983,0.061079,...,-0.013950,-0.027494,0.034615,-0.032077,-0.051153,-0.004736,-0.004042,-0.052157,-0.026376,2026-03-06 13:00:00
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077088,-0.078418,0.091533,...,0.025732,0.000545,0.020312,-0.028054,0.001602,0.006979,-0.064937,0.022653,-0.017796,2026-03-06 12:00:00
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053648,-0.076999,0.085929,...,-0.003562,-0.043411,-0.007184,-0.009170,0.006113,-0.004557,0.020586,0.010520,0.012833,2026-03-06 08:00:00
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059695,...,-0.007507,-0.021414,0.028226,-0.025335,-0.045774,-0.001725,0.002281,-0.042259,-0.017379,2026-03-05 16:00:00
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089563,-0.088812,0.109835,...,0.027449,-0.019751,0.008681,0.023138,-0.009474,-0.030269,0.013652,-0.022735,0.006830,2026-03-05 15:00:00


In [27]:
tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)

In [28]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [29]:
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_final.shape

(634680, 473)

In [31]:
df_final.sample(15)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
627888,2024-05-23 00:00:00,26,50.45060,30.52430,20.6,11.0,56.3,8.000,100.0,4.17,...,-0.007644,0.021627,-0.001459,0.001631,0.007930,-0.012310,0.005647,-0.006674,-0.014783,-0.010977
244798,2022-12-04 02:00:00,11,48.50850,32.26560,-5.1,-10.4,66.6,0.000,0.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223710,2023-07-15 08:00:00,10,50.45060,30.52430,21.2,11.9,58.3,0.000,0.0,0.00,...,0.001956,-0.024918,-0.038226,-0.015839,-0.010359,0.032616,0.036935,-0.008549,-0.014687,0.026583
385020,2023-11-02 08:00:00,17,50.61930,26.25130,9.1,5.3,79.1,0.000,0.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81960,2022-06-13 10:00:00,5,48.00200,37.81450,25.2,11.5,44.6,0.100,100.0,4.17,...,0.005077,0.014198,-0.012112,0.006441,-0.014683,0.003262,0.002692,-0.005065,0.007362,0.002545
605407,2024-11-04 04:00:00,25,51.49370,31.29440,3.6,-3.1,62.1,0.400,100.0,4.17,...,0.004744,-0.016538,0.003490,0.005876,-0.007340,-0.011628,-0.000891,-0.003329,-0.004476,0.015131
212378,2022-03-30 03:00:00,10,50.45060,30.52430,8.4,3.0,69.8,0.300,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262844,2024-12-25 02:00:00,11,48.50850,32.26560,2.2,1.4,94.7,0.200,100.0,8.33,...,0.000098,0.010948,0.003981,0.010092,-0.005002,0.004189,-0.009527,0.001242,-0.003597,0.002071
43121,2024-01-19 22:00:00,3,50.74690,25.32630,-1.8,-2.9,92.1,1.634,100.0,4.17,...,-0.013623,0.000820,-0.011414,-0.007846,-0.007731,-0.009413,0.019898,-0.013036,0.019832,-0.007785
344257,2022-03-15 16:00:00,16,49.58790,34.55170,0.5,-5.1,67.2,0.000,0.0,0.00,...,-0.002241,0.000485,-0.012129,-0.007545,-0.008257,-0.001473,0.000290,-0.002247,-0.004348,0.008489
